#  Phân Tích Đánh Giá Khách Sạn Bằng Apache Pig

### Mô tả dữ liệu

| Cột | Tên | Mô tả | Ví dụ |
|-----|-----|-------|-------|
| 1 | `id` | Mã bình luận | 1, 2, 3... |
| 2 | `review` | Nội dung bình luận | "Phòng sạch sẽ, nhân viên thân thiện" |
| 3 | `category` | Phân loại đánh giá | GENERAL, QUALITY, DESIGN&FEATURES... |
| 4 | `aspect` | Khía cạnh đánh giá | HOTEL, SERVICE, ROOMS, FOOD&DRINKS... |
| 5 | `sentiment` | Cảm xúc | positive, negative, neutral |

**Phân cách:** dấu chấm phẩy (`;`)

### Mục tiêu
1. Tiền xử lý dữ liệu văn bản tiếng Việt
2. Thống kê tần số từ, bình luận theo category và aspect
3. Xác định khía cạnh tích cực/tiêu cực nhất
4. Tìm top từ tích cực/tiêu cực theo từng phân loại
5. Tìm top từ liên quan nhất theo từng phân loại


In [1]:
%%bash
export PIG_HOME=/home/nbl2005/pig-0.17.0
export PATH=$PATH:$PIG_HOME/bin
export JAVA_HOME=$(dirname $(dirname $(readlink -f $(which java))))

pig --version 2>&1 | grep "Apache Pig version"
echo "Pig OK!"


Apache Pig version 0.17.0 (r1797386) 
Pig OK!


## Bài 1:

Thực hiện các thao tác sau đối với bộ dữ liệu:
- Đưa tất cả ký tự về chữ thường (lowercase).
- Tách các dòng bình luận thành dãy các từ (từ được tách ra từ câu theo khoảng trắng).
- Loại bỏ các stop word (dựa vào danh sách ```stopword.txt```)



In [39]:
%%bash
export PIG_HOME=/home/nbl2005/pig-0.17.0
export PATH=$PATH:$PIG_HOME/bin
export JAVA_HOME=$(dirname $(dirname $(readlink -f $(which java))))

cd /home/nbl2005/DS200.Q21.1.LAB/DS200.Q21.1.Lab02

cat > Source_Pig/bai1.pig <<'EOF'

raw = LOAD 'Data/hotel-review.csv' 
      USING PigStorage(';') 
      AS (id:int, review:chararray, category:chararray, aspect:chararray, sentiment:chararray);

stopwords = LOAD 'Data/stopwords.txt' AS (word:chararray);

--  Clean text (giữ chữ + space)
clean_review = FOREACH raw GENERATE 
    id,
    REPLACE(LOWER(review), '[^a-zà-ỹ\\s]', '') AS review,
    category, aspect, sentiment;

--  Tokenize
tokens = FOREACH clean_review GENERATE 
    id,
    FLATTEN(TOKENIZE(review)) AS word,
    category, aspect, sentiment;

--  Clean word (fix \r\n + lowercase lại)
tokens_clean = FOREACH tokens GENERATE 
    id,
    LOWER(TRIM(REPLACE(word, '\\r|\\n', ''))) AS word,
    category, aspect, sentiment;

tokens_clean = FILTER tokens_clean BY 
    word IS NOT NULL AND word != '';

--  Stopwords clean
stops = FOREACH stopwords GENERATE 
    LOWER(TRIM(REPLACE(word, '\\r|\\n', ''))) AS stopword;

-- Remove stopwords
joined = JOIN tokens_clean BY word LEFT OUTER, stops BY stopword;
filtered = FILTER joined BY stops::stopword IS NULL;

-- OUTPUT
bai1_out = FOREACH filtered GENERATE 
    tokens_clean::id,
    tokens_clean::word,
    tokens_clean::category,
    tokens_clean::aspect,
    tokens_clean::sentiment;

STORE bai1_out INTO 'Output/BaiTap1' USING PigStorage('\t');
EOF

In [40]:
%%bash
export PIG_HOME=/home/nbl2005/pig-0.17.0
export PATH=$PATH:$PIG_HOME/bin
export JAVA_HOME=$(dirname $(dirname $(readlink -f $(which java))))

cd /home/nbl2005/DS200.Q21.1.LAB/DS200.Q21.1.Lab02

# Dọn dẹp thư mục cũ và chạy Pig
rm -rf Output/BaiTap1
pig -x local Source_Pig/bai1.pig

echo " 10 DÒNG ĐẦU KẾT QUẢ BÀI 1"
head -10 Output/BaiTap1/part-r-00000

2026-04-09 05:05:17,538 INFO pig.ExecTypeProvider: Trying ExecType : LOCAL
2026-04-09 05:05:17,538 INFO pig.ExecTypeProvider: Picked LOCAL as the ExecType
2026-04-09 05:05:17,743 [main] INFO  org.apache.pig.Main - Apache Pig version 0.17.0 (r1797386) compiled Jun 02 2017, 15:41:58
2026-04-09 05:05:17,744 [main] INFO  org.apache.pig.Main - Logging error messages to: /home/nbl2005/DS200.Q21.1.LAB/DS200.Q21.1.Lab02/pig_1775711117735.log
2026-04-09 05:05:17,803 [main] INFO  org.apache.hadoop.conf.Configuration.deprecation - user.name is deprecated. Instead, use mapreduce.job.user.name
2026-04-09 05:05:18,162 [main] INFO  org.apache.pig.impl.util.Utils - Default bootup file /home/nbl2005/.pigbootup not found
2026-04-09 05:05:18,293 [main] INFO  org.apache.hadoop.conf.Configuration.deprecation - mapred.job.tracker is deprecated. Instead, use mapreduce.jobtracker.address
2026-04-09 05:05:18,298 [main] INFO  org.apache.pig.backend.hadoop.executionengine.HExecutionEngine - Connecting to hadoop 

 10 DÒNG ĐẦU KẾT QUẢ BÀI 1
6889	a	MISCELLANEOUS	ROOM_AMENITIES	negative
1730	a	MISCELLANEOUS	ROOM_AMENITIES	negative
4177	b	GENERAL	SERVICE	positive
4177	b	GENERAL	LOCATION	positive
4177	b	GENERAL	HOTEL	positive
4177	b	CLEANLINESS	ROOMS	positive
6889	c	MISCELLANEOUS	ROOM_AMENITIES	negative
1151	g	GENERAL	SERVICE	positive
5061	h	GENERAL	HOTEL	negative
546	h	COMFORT	FACILITIES	positive


---
##  Bài 2: Thống kê

**Yêu cầu:**
- Thống kê tần số xuất hiện của các từ → Chỉ ra các từ xuất hiện trên 500 lần
- Thống kê số bình luận theo từng phân loại (category)
- Thống kê số bình luận theo từng khía cạnh đánh giá (aspect)



In [41]:
%%bash
export PIG_HOME=/home/nbl2005/pig-0.17.0
export PATH=$PATH:$PIG_HOME/bin
export JAVA_HOME=$(dirname $(dirname $(readlink -f $(which java))))

cd /home/nbl2005/DS200.Q21.1.LAB/DS200.Q21.1.Lab02

cat > Source_Pig/bai2.pig <<'EOF'
raw = LOAD 'Data/hotel-review.csv' USING PigStorage(';') 
AS (id:int, review:chararray, category:chararray, aspect:chararray, sentiment:chararray);

bai1 = LOAD 'Output/BaiTap1' USING PigStorage('\t') 
AS (id:int, word:chararray, category:chararray, aspect:chararray, sentiment:chararray);

-- 2a. WORD FREQ > 500
grp_word = GROUP bai1 BY word;
word_freq = FOREACH grp_word GENERATE group AS word, COUNT(bai1) AS freq;
word_filtered = FILTER word_freq BY freq > 500;
STORE (ORDER word_filtered BY freq DESC) INTO 'Output/BaiTap2_WordFreq' USING PigStorage('\t');

-- 2b. CATEGORY COUNT
grp_cat = GROUP raw BY category;
cat_count = FOREACH grp_cat GENERATE group AS category, COUNT(raw) AS total;
STORE (ORDER cat_count BY total DESC) INTO 'Output/BaiTap2_Category' USING PigStorage('\t');

-- 2c. ASPECT COUNT
grp_asp = GROUP raw BY aspect;
asp_count = FOREACH grp_asp GENERATE group AS aspect, COUNT(raw) AS total;
STORE (ORDER asp_count BY total DESC) INTO 'Output/BaiTap2_Aspect' USING PigStorage('\t');
EOF

In [42]:
%%bash
export PIG_HOME=/home/nbl2005/pig-0.17.0
export PATH=$PATH:$PIG_HOME/bin
export JAVA_HOME=$(dirname $(dirname $(readlink -f $(which java))))

cd /home/nbl2005/DS200.Q21.1.LAB/DS200.Q21.1.Lab02

rm -rf Output/BaiTap2_*
pig -x local /home/nbl2005/DS200.Q21.1.LAB/DS200.Q21.1.Lab02/Source_Pig/bai2.pig 2>&1 | tail -5

echo "  2a. CÁC TỪ XUẤT HIỆN TRÊN 500 LẦN"
echo ""
printf "%-30s %s\n" "TỪ" "TẦN SỐ"

cat Output/BaiTap2_WordFreq/part-r-00000 | while IFS=$'\t' read word freq; do
    printf "%-30s %s\n" "$word" "$freq"
done

echo ""
echo "  2b. SỐ BÌNH LUẬN THEO CATEGORY"
echo ""
printf "%-25s %s\n" "CATEGORY" "SỐ BÌNH LUẬN"

cat Output/BaiTap2_Category/part-r-00000 | while IFS=$'\t' read cat total; do
    printf "%-25s %s\n" "$cat" "$total"
done

echo ""
echo "  2c. SỐ BÌNH LUẬN THEO ASPECT"
echo ""
printf "%-25s %s\n" "ASPECT" "SỐ BÌNH LUẬN"

cat Output/BaiTap2_Aspect/part-r-00000 | while IFS=$'\t' read asp total; do
    printf "%-25s %s\n" "$asp" "$total"
done

2026-04-09 05:06:24,419 [main] WARN  org.apache.hadoop.metrics2.impl.MetricsSystemImpl - JobTracker metrics system already initialized!
2026-04-09 05:06:24,421 [main] WARN  org.apache.hadoop.metrics2.impl.MetricsSystemImpl - JobTracker metrics system already initialized!
2026-04-09 05:06:24,422 [main] WARN  org.apache.hadoop.metrics2.impl.MetricsSystemImpl - JobTracker metrics system already initialized!
2026-04-09 05:06:24,425 [main] INFO  org.apache.pig.backend.hadoop.executionengine.mapReduceLayer.MapReduceLauncher - Success!
2026-04-09 05:06:24,463 [main] INFO  org.apache.pig.Main - Pig script completed in 14 seconds and 229 milliseconds (14229 ms)
  2a. CÁC TỪ XUẤT HIỆN TRÊN 500 LẦN

TỪ                           TẦN SỐ
phòng                         6732
khách                         5190
sạn                          4059
vụ                           3811
tiện                         3277
không                         3260
nhân                          3209
viên                    

---
##  Bài 3: Khía cạnh tích cực nhất & tiêu cực nhất

**Yêu cầu:**  
Xác định khía cạnh (aspect) nào nhận nhiều đánh giá tiêu cực (negative) nhất, 
và khía cạnh nào nhận nhiều đánh giá tích cực (positive) nhất.


In [58]:
%%bash
export PIG_HOME=/home/nbl2005/pig-0.17.0
export PATH=$PATH:$PIG_HOME/bin
export JAVA_HOME=$(dirname $(dirname $(readlink -f $(which java))))

cd /home/nbl2005/DS200.Q21.1.LAB/DS200.Q21.1.Lab02

cat > Source_Pig/bai3.pig <<'EOF'
raw = LOAD 'Data/hotel-review.csv' USING PigStorage(';') 
AS (id:int, review:chararray, category:chararray, aspect:chararray, sentiment:chararray);

clean = FOREACH raw GENERATE 
    id,
    review,
    category,
    aspect,
    LOWER(TRIM(REPLACE(sentiment, '\\r|\\n', ''))) AS sentiment;

-- NEGATIVE
neg = FILTER clean BY sentiment == 'negative';
grp_neg = GROUP neg BY aspect;
cnt_neg = FOREACH grp_neg GENERATE group AS aspect, COUNT(neg) AS total;
sorted_neg = ORDER cnt_neg BY total DESC;

STORE sorted_neg INTO 'Output/BaiTap3_NegativeAll' USING PigStorage('\t');

-- POSITIVE
pos = FILTER clean BY sentiment == 'positive';
grp_pos = GROUP pos BY aspect;
cnt_pos = FOREACH grp_pos GENERATE group AS aspect, COUNT(pos) AS total;
sorted_pos = ORDER cnt_pos BY total DESC;

STORE sorted_pos INTO 'Output/BaiTap3_PositiveAll' USING PigStorage('\t');
EOF

In [59]:
%%bash
export PIG_HOME=/home/nbl2005/pig-0.17.0
export PATH=$PATH:$PIG_HOME/bin
export JAVA_HOME=$(dirname $(dirname $(readlink -f $(which java))))

cd /home/nbl2005/DS200.Q21.1.LAB/DS200.Q21.1.Lab02

rm -rf Output/BaiTap3_TopNegative
rm -rf Output/BaiTap3_TopPositive


pig -x local Source_Pig/bai3.pig

echo "NEGATIVE TOP"
head -1 Output/BaiTap3_NegativeAll/part-r-00000

echo "POSITIVE TOP"
head -1 Output/BaiTap3_PositiveAll/part-r-00000

2026-04-09 05:32:55,887 INFO pig.ExecTypeProvider: Trying ExecType : LOCAL
2026-04-09 05:32:55,888 INFO pig.ExecTypeProvider: Picked LOCAL as the ExecType
2026-04-09 05:32:56,031 [main] INFO  org.apache.pig.Main - Apache Pig version 0.17.0 (r1797386) compiled Jun 02 2017, 15:41:58
2026-04-09 05:32:56,031 [main] INFO  org.apache.pig.Main - Logging error messages to: /home/nbl2005/DS200.Q21.1.LAB/DS200.Q21.1.Lab02/pig_1775712776025.log
2026-04-09 05:32:56,066 [main] INFO  org.apache.hadoop.conf.Configuration.deprecation - user.name is deprecated. Instead, use mapreduce.job.user.name
2026-04-09 05:32:56,318 [main] INFO  org.apache.pig.impl.util.Utils - Default bootup file /home/nbl2005/.pigbootup not found
2026-04-09 05:32:56,445 [main] INFO  org.apache.hadoop.conf.Configuration.deprecation - mapred.job.tracker is deprecated. Instead, use mapreduce.jobtracker.address
2026-04-09 05:32:56,450 [main] INFO  org.apache.pig.backend.hadoop.executionengine.HExecutionEngine - Connecting to hadoop 

NEGATIVE TOP
ROOMS	515
POSITIVE TOP
HOTEL	2942


---
### Bài 4: Top 5 từ tích cực / tiêu cực theo từng category

**Yêu cầu:**
- Dựa vào từng phân loại bình luận, xác định **5 từ** mang ý nghĩa tích cực nhất
- Dựa vào từng phân loại bình luận, xác định **5 từ** mang ý nghĩa tiêu cực nhất



In [61]:
%%bash
export PIG_HOME=/home/nbl2005/pig-0.17.0
export PATH=$PATH:$PIG_HOME/bin
export JAVA_HOME=$(dirname $(dirname $(readlink -f $(which java))))

cd /home/nbl2005/DS200.Q21.1.LAB/DS200.Q21.1.Lab02

cat > Source_Pig/bai4.pig <<'EOF'
data = LOAD 'Output/BaiTap1' USING PigStorage('\t') 
AS (id:int, word:chararray, category:chararray, aspect:chararray, sentiment:chararray);

-- POSITIVE
pos = FILTER data BY sentiment == 'positive';

grp_pos = GROUP pos BY (category, word);
cnt_pos = FOREACH grp_pos GENERATE 
    FLATTEN(group) AS (category, word), 
    COUNT(pos) AS freq;

grp_cat_pos = GROUP cnt_pos BY category;

top_pos = FOREACH grp_cat_pos {
    sorted = ORDER cnt_pos BY freq DESC;
    top5 = LIMIT sorted 5;
    GENERATE group AS category, FLATTEN(top5);
};

STORE top_pos INTO 'Output/BaiTap4_Positive' USING PigStorage('\t');


-- NEGATIVE
neg = FILTER data BY sentiment == 'negative';

grp_neg = GROUP neg BY (category, word);
cnt_neg = FOREACH grp_neg GENERATE 
    FLATTEN(group) AS (category, word), 
    COUNT(neg) AS freq;

grp_cat_neg = GROUP cnt_neg BY category;

top_neg = FOREACH grp_cat_neg {
    sorted = ORDER cnt_neg BY freq DESC;
    top5 = LIMIT sorted 5;
    GENERATE group AS category, FLATTEN(top5);
};

STORE top_neg INTO 'Output/BaiTap4_Negative' USING PigStorage('\t');
EOF

In [62]:
%%bash
export PIG_HOME=/home/nbl2005/pig-0.17.0
export PATH=$PATH:$PIG_HOME/bin
export JAVA_HOME=$(dirname $(dirname $(readlink -f $(which java))))

cd /home/nbl2005/DS200.Q21.1.LAB/DS200.Q21.1.Lab02

rm -rf Output/BaiTap4_*

pig -x local Source_Pig/bai4.pig

echo "TOP POSITIVE (sample)"
head -20 Output/BaiTap4_Positive/part-r-00000

echo ""
echo "TOP NEGATIVE (sample)"
head -20 Output/BaiTap4_Negative/part-r-00000

2026-04-09 05:35:03,857 INFO pig.ExecTypeProvider: Trying ExecType : LOCAL
2026-04-09 05:35:03,858 INFO pig.ExecTypeProvider: Picked LOCAL as the ExecType
2026-04-09 05:35:03,982 [main] INFO  org.apache.pig.Main - Apache Pig version 0.17.0 (r1797386) compiled Jun 02 2017, 15:41:58
2026-04-09 05:35:03,983 [main] INFO  org.apache.pig.Main - Logging error messages to: /home/nbl2005/DS200.Q21.1.LAB/DS200.Q21.1.Lab02/pig_1775712903979.log
2026-04-09 05:35:04,025 [main] INFO  org.apache.hadoop.conf.Configuration.deprecation - user.name is deprecated. Instead, use mapreduce.job.user.name
2026-04-09 05:35:04,321 [main] INFO  org.apache.pig.impl.util.Utils - Default bootup file /home/nbl2005/.pigbootup not found
2026-04-09 05:35:04,499 [main] INFO  org.apache.hadoop.conf.Configuration.deprecation - mapred.job.tracker is deprecated. Instead, use mapreduce.jobtracker.address
2026-04-09 05:35:04,502 [main] INFO  org.apache.pig.backend.hadoop.executionengine.HExecutionEngine - Connecting to hadoop 

TOP POSITIVE (sample)
PRICES	PRICES	giá	602
PRICES	PRICES	cả	411
PRICES	PRICES	hợp	320
PRICES	PRICES	vụ	312
PRICES	PRICES	phòng	271
COMFORT	COMFORT	lòng	541
COMFORT	COMFORT	hài	530
COMFORT	COMFORT	khách	440
COMFORT	COMFORT	phòng	436
COMFORT	COMFORT	sạn	382
GENERAL	GENERAL	khách	2426
GENERAL	GENERAL	phòng	2249
GENERAL	GENERAL	vụ	2024
GENERAL	GENERAL	nhân	1848
GENERAL	GENERAL	sạn	1827
QUALITY	QUALITY	phòng	332
QUALITY	QUALITY	chất	326
QUALITY	QUALITY	lượng	312
QUALITY	QUALITY	khách	269
QUALITY	QUALITY	sạn	235

TOP NEGATIVE (sample)
PRICES	PRICES	giá	56
PRICES	PRICES	không	47
PRICES	PRICES	phòng	30
PRICES	PRICES	tiền	23
PRICES	PRICES	khách	21
COMFORT	COMFORT	phòng	216
COMFORT	COMFORT	không	197
COMFORT	COMFORT	khách	107
COMFORT	COMFORT	sạn	81
COMFORT	COMFORT	lòng	71
GENERAL	GENERAL	không	371
GENERAL	GENERAL	phòng	338
GENERAL	GENERAL	khách	274
GENERAL	GENERAL	sạn	199
GENERAL	GENERAL	nhân	150
QUALITY	QUALITY	không	276
QUALITY	QUALITY	phòng	246
QUALITY	QUALITY	cũ	158
QUALITY	QUALITY	khách	144

## Bài 5: Top 5 từ liên quan nhất theo từng category
Dựa vào từng phân loại bình luận, xác định **5 từ** liên quan nhất 
(từ xuất hiện nhiều nhất trong category đó, không phân biệt sentiment).


In [63]:
%%bash
export PIG_HOME=/home/nbl2005/pig-0.17.0
export PATH=$PATH:$PIG_HOME/bin
export JAVA_HOME=$(dirname $(dirname $(readlink -f $(which java))))

cd /home/nbl2005/DS200.Q21.1.LAB/DS200.Q21.1.Lab02

cat > Source_Pig/bai5.pig <<'EOF'

data = LOAD 'Output/BaiTap1' USING PigStorage('\t') 
AS (id:int, word:chararray, category:chararray, aspect:chararray, sentiment:chararray);

-- Đếm tần số theo (category, word)
grp = GROUP data BY (category, word);

cnt = FOREACH grp GENERATE 
    FLATTEN(group) AS (category, word),
    COUNT(data) AS freq;

-- Nhóm theo category
grp_cat = GROUP cnt BY category;

-- Lấy top 5 mỗi category
top5 = FOREACH grp_cat {
    sorted = ORDER cnt BY freq DESC;
    top = LIMIT sorted 5;
    GENERATE group AS category, FLATTEN(top);
};

STORE top5 INTO 'Output/BaiTap5' USING PigStorage('\t');

bash: line 29: warning: here-document at line 7 delimited by end-of-file (wanted `EOF')


In [64]:
%%bash
export PIG_HOME=/home/nbl2005/pig-0.17.0
export PATH=$PATH:$PIG_HOME/bin
export JAVA_HOME=$(dirname $(dirname $(readlink -f $(which java))))

cd /home/nbl2005/DS200.Q21.1.LAB/DS200.Q21.1.Lab02

rm -rf Output/BaiTap5

pig -x local Source_Pig/bai5.pig

echo "TOP WORDS PER CATEGORY (sample)"
head -20 Output/BaiTap5/part-r-00000

2026-04-09 05:35:42,237 INFO pig.ExecTypeProvider: Trying ExecType : LOCAL
2026-04-09 05:35:42,239 INFO pig.ExecTypeProvider: Picked LOCAL as the ExecType
2026-04-09 05:35:42,393 [main] INFO  org.apache.pig.Main - Apache Pig version 0.17.0 (r1797386) compiled Jun 02 2017, 15:41:58
2026-04-09 05:35:42,393 [main] INFO  org.apache.pig.Main - Logging error messages to: /home/nbl2005/DS200.Q21.1.LAB/DS200.Q21.1.Lab02/pig_1775712942390.log
2026-04-09 05:35:42,438 [main] INFO  org.apache.hadoop.conf.Configuration.deprecation - user.name is deprecated. Instead, use mapreduce.job.user.name
2026-04-09 05:35:42,682 [main] INFO  org.apache.pig.impl.util.Utils - Default bootup file /home/nbl2005/.pigbootup not found
2026-04-09 05:35:42,819 [main] INFO  org.apache.hadoop.conf.Configuration.deprecation - mapred.job.tracker is deprecated. Instead, use mapreduce.jobtracker.address
2026-04-09 05:35:42,825 [main] INFO  org.apache.pig.backend.hadoop.executionengine.HExecutionEngine - Connecting to hadoop 

TOP WORDS PER CATEGORY (sample)
PRICES	PRICES	giá	697
PRICES	PRICES	cả	451
PRICES	PRICES	vụ	338
PRICES	PRICES	hợp	334
PRICES	PRICES	phòng	320
COMFORT	COMFORT	phòng	657
COMFORT	COMFORT	lòng	624
COMFORT	COMFORT	hài	612
COMFORT	COMFORT	khách	552
COMFORT	COMFORT	không	504
GENERAL	GENERAL	khách	2882
GENERAL	GENERAL	phòng	2750
GENERAL	GENERAL	vụ	2296
GENERAL	GENERAL	sạn	2185
GENERAL	GENERAL	nhân	2101
QUALITY	QUALITY	phòng	608
QUALITY	QUALITY	khách	454
QUALITY	QUALITY	chất	452
QUALITY	QUALITY	lượng	417
QUALITY	QUALITY	không	406


In [65]:
%%bash
cd /home/nbl2005/DS200.Q21.1.LAB/DS200.Q21.1.Lab02

mkdir -p Result

echo "ĐANG CHUYỂN KẾT QUẢ SANG FILE TXT..."

copy_if_exists () {
    if ls $1 1> /dev/null 2>&1; then
        cat $1 > $2
    else
        echo " Missing: $1"
    fi
}

# Bài 1
copy_if_exists "Output/BaiTap1/part-*" "Result/Bai1_TienXuLy.txt"

# Bài 2
copy_if_exists "Output/BaiTap2_WordFreq/part-*" "Result/Bai2a_WordFreq.txt"
copy_if_exists "Output/BaiTap2_Category/part-*" "Result/Bai2b_Category.txt"
copy_if_exists "Output/BaiTap2_Aspect/part-*" "Result/Bai2c_Aspect.txt"

# Bài 3
copy_if_exists "Output/BaiTap3_PositiveAll/part-*" "Result/Bai3a_Positive.txt"
copy_if_exists "Output/BaiTap3_NegativeAll/part-*" "Result/Bai3b_Negative.txt"

# Bài 4
copy_if_exists "Output/BaiTap4_Positive/part-*" "Result/Bai4a_Top5_Positive.txt"
copy_if_exists "Output/BaiTap4_Negative/part-*" "Result/Bai4b_Top5_Negative.txt"

# Bài 5
copy_if_exists "Output/BaiTap5/part-*" "Result/Bai5_Top5_Related.txt"

echo ""
echo "DONE! DANH SÁCH FILE RESULT:"
echo ""

ls -lh Result

ĐANG CHUYỂN KẾT QUẢ SANG FILE TXT...

DONE! DANH SÁCH FILE RESULT:

total 6.4M
-rw-r--r-- 1 nbl2005 nbl2005 6.4M Apr  9 05:36 Bai1_TienXuLy.txt
-rw-r--r-- 1 nbl2005 nbl2005  833 Apr  9 05:36 Bai2a_WordFreq.txt
-rw-r--r-- 1 nbl2005 nbl2005  123 Apr  9 05:36 Bai2b_Category.txt
-rw-r--r-- 1 nbl2005 nbl2005   99 Apr  9 05:36 Bai2c_Aspect.txt
-rw-r--r-- 1 nbl2005 nbl2005   98 Apr  9 05:36 Bai3a_Positive.txt
-rw-r--r-- 1 nbl2005 nbl2005   94 Apr  9 05:36 Bai3b_Negative.txt
-rw-r--r-- 1 nbl2005 nbl2005 1.3K Apr  9 05:36 Bai4a_Top5_Positive.txt
-rw-r--r-- 1 nbl2005 nbl2005 1.3K Apr  9 05:36 Bai4b_Top5_Negative.txt
-rw-r--r-- 1 nbl2005 nbl2005 1.3K Apr  9 05:36 Bai5_Top5_Related.txt


In [1]:
%%bash

echo " THỰC HÀNH PIG - PHÂN TÍCH DỮ LIỆU HOTEL REVIEW"
echo " Sinh viên: Nguyễn Bá Long - MSSV: 23520880"
echo " User hệ thống: $USER@$(hostname)"
echo " Đường dẫn hiện tại: $(pwd)"

echo " ĐÃ LƯU THÀNH CÔNG TẤT CẢ KẾT QUẢ VÀO FOLDER RESULT!"
echo -e "\n Danh sách các file kết quả:"

# Chỉ liệt kê tên file và dung lượng, KHÔNG in nội dung bên trong
ls -lh /home/nbl2005/DS200.Q21.1.LAB/DS200.Q21.1.Lab02/Result/

 THỰC HÀNH PIG - PHÂN TÍCH DỮ LIỆU HOTEL REVIEW
 Sinh viên: Nguyễn Bá Long - MSSV: 23520880
 User hệ thống: nbl2005@AiCiCi
 Đường dẫn hiện tại: /home/nbl2005/DS200.Q21.1.LAB/DS200.Q21.1.Lab02/Notebook
 ĐÃ LƯU THÀNH CÔNG TẤT CẢ KẾT QUẢ VÀO FOLDER RESULT!

 Danh sách các file kết quả:
total 6.5M
-rw-r--r-- 1 nbl2005 nbl2005 6.4M Apr  6 03:11 Bai1_TienXuLy.txt
-rw-r--r-- 1 nbl2005 nbl2005   99 Apr  6 03:11 Bai2a_Aspect.txt
-rw-r--r-- 1 nbl2005 nbl2005  123 Apr  6 03:11 Bai2b_Category.txt
-rw-r--r-- 1 nbl2005 nbl2005  839 Apr  6 03:11 Bai2c_WordFreq.txt
-rw-r--r-- 1 nbl2005 nbl2005   98 Apr  6 03:11 Bai3a_PositiveAll.txt
-rw-r--r-- 1 nbl2005 nbl2005   94 Apr  6 03:11 Bai3b_NegativeAll.txt
-rw-r--r-- 1 nbl2005 nbl2005  840 Apr  6 03:11 Bai4a_Top5_Pos.txt
-rw-r--r-- 1 nbl2005 nbl2005  834 Apr  6 03:11 Bai4b_Top5_Neg.txt
-rw-r--r-- 1 nbl2005 nbl2005  857 Apr  6 03:11 Bai5_Top5_Related.txt
